# Predicting arrears next month
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), 5-fold walk-forward CV on the training rows, choice by mean CV ROC AUC, holdout scored once.

`final_notice_sent` is **excluded**: per `columns.csv` it is recorded in the billing cycle *after* the billing month, i.e. it is a consequence of the outcome and not known at prediction time (arrears rate 87% vs 1.4% with/without it). `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
is_hold = df.index.isin(hold_idx)
train, hold = df[~is_hold], df[is_hold]
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min())

In [ ]:
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
# all candidates named before any is scored
candidates = {
 'baseline_prior': DummyClassifier(strategy='prior'),
 'logreg_C1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'random_forest': Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hist_gb': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
list(candidates)

In [ ]:
# walk-forward CV over months: expanding window, 5 folds, inside training rows only
months = np.sort(train.billing_month.unique()); K=5
size = len(months)//(K+1)
blocks = [months[len(months)-size*(K-i):len(months)-size*(K-i-1)] for i in range(K)]  # last K blocks validate
folds=[]
for b in blocks:
    tr = np.where(train.billing_month.values < b[0])[0]
    va = np.where(train.billing_month.isin(b).values)[0]
    folds.append((tr,va))
    print(b[0], b[-1], len(tr), len(va))
X, y = train[FEATURES], train[TARGET].to_numpy()
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,va in folds:
        f=clone(m).fit(X.iloc[tr],y[tr])   # preprocessing is inside the pipeline, fit per fold
        s.append(roc_auc_score(y[va],f.predict_proba(X.iloc[va])[:,1]))
    cv_scores[name]=s
cvdf=pd.DataFrame(cv_scores).T; cvdf['mean']=cvdf.mean(axis=1); cvdf.sort_values('mean',ascending=False)

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
final = clone(candidates[chosen]).fit(X,y)
hold_auc = roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1])
print(chosen, 'CV', np.mean(cv_scores[chosen]), 'holdout', hold_auc)

In [ ]:
train_pos = df.index[~is_hold].to_numpy()
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[train_pos[tr].tolist(), train_pos[va].tolist()] for tr,va in folds],
 'cv_scores': {k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen': chosen,
 'holdout_rows': hold.index.tolist(),
 'holdout_scores': {chosen: float(hold_auc)},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the CV table and the holdout ROC AUC printed above. The headline is the holdout ROC AUC of the CV-chosen candidate, scored once. Without `final_notice_sent` (a post-outcome leak), the predictive signal comes only from tariff, usage, bill and late-payment history.